# Exercise 06 — Vectors, Arrays and Matrices

NUMA01 VT2026 — Arvid Brenner & Sixten Midsem

Lösningar till `exercise06.md`. Grunder i NumPy: matris-vektorprodukter,
symmetri, ortogonalitet, normalisering, rotationer, egenvärden.


In [1]:
import numpy as np
from numpy.linalg import norm as scnorm
import matplotlib.pyplot as plt


## Task 2 — Matris-vektorprodukt

`@` är matris-multiplikationsoperatorn. För `M @ v` med
`M.shape == (3, 3)` och `v.shape == (3,)` får vi $z_i = \sum_j M_{ij} v_j$.


In [2]:
v = np.array([1., 2., 3.])
M = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 9.]])
z = M @ v
print("z =", z)   # [14, 32, 50]


z = [14. 32. 50.]


## Task 3 — Symmetri och skev-symmetri

Returnerar `1` om $A = A^T$, `-1` om $A = -A^T$, annars `0`. `np.allclose`
tål små flyttalsfel.


In [3]:
def symmetry(A):
    """Returnera 1 om A är symmetrisk, -1 om skev-symmetrisk, annars 0."""
    if A.shape[0] != A.shape[1]:
        return 0
    if np.allclose(A, A.T):
        return 1
    if np.allclose(A, -A.T):
        return -1
    return 0


S = np.array([[1., 2, 3], [2, 4, 5], [3, 5, 6]])
K = np.array([[0., 2, -3], [-2, 0, 5], [3, -5, 0]])
N = np.array([[1., 2, 3], [4, 5, 6], [7, 8, 9]])

print("symmetrisk    →", symmetry(S))   # 1
print("skev-symm.    →", symmetry(K))   # -1
print("varken/eller  →", symmetry(N))   # 0


symmetrisk    → 1
skev-symm.    → -1
varken/eller  → 0


## Task 4 — Ortogonalitet

Två vektorer är ortogonala om deras skalärprodukt är noll. `np.isclose`
hanterar avrundningsfel.


In [4]:
def orthogonal(u, v):
    """Returnera True om u och v är ortogonala (skalärprodukt 0)."""
    return bool(np.isclose(np.dot(u, v), 0.0))


print(orthogonal(np.array([1, 0, 0]), np.array([0, 1, 0])))   # True
print(orthogonal(np.array([1, 2, 3]), np.array([1, 0, 0])))   # False
print(orthogonal(np.array([1, 1]),    np.array([1, -1])))     # True


True
False
True


## Task 5 — Normalisering

Två varianter: en med egen norm-beräkning, en med `numpy.linalg.norm`
(importerad som `scnorm`).


In [5]:
def normalize_manual(x):
    """Normalisera x med egen 2-norm."""
    n = np.sqrt(np.sum(x ** 2))
    return x / n


def normalize_np(x):
    """Normalisera x med numpy.linalg.norm."""
    return x / scnorm(x)


x = np.array([3., 4.])
print("manual:", normalize_manual(x))   # [0.6, 0.8]
print("numpy: ", normalize_np(x))
print("|y|  : ", scnorm(normalize_np(x)))   # 1.0


manual: [0.6 0.8]
numpy:  [0.6 0.8]
|y|  :  1.0


## Task 6 — Rotationsmatrisens invers är dess transponat

$R(\alpha)^T R(\alpha) = I$ för alla $\alpha$. Visas experimentellt.


In [6]:
def R(alpha):
    return np.array([[np.cos(alpha),  np.sin(alpha)],
                     [-np.sin(alpha), np.cos(alpha)]])


for alpha in [0, np.pi/6, np.pi/4, 1.234, -2.5]:
    A = R(alpha)
    I1 = A @ A.T
    I2 = A.T @ A
    ok = np.allclose(I1, np.eye(2)) and np.allclose(I2, np.eye(2))
    print(f"alpha = {alpha:7.4f}  →  A A^T ≈ I, A^T A ≈ I  →  {ok}")


alpha =  0.0000  →  A A^T ≈ I, A^T A ≈ I  →  True
alpha =  0.5236  →  A A^T ≈ I, A^T A ≈ I  →  True
alpha =  0.7854  →  A A^T ≈ I, A^T A ≈ I  →  True
alpha =  1.2340  →  A A^T ≈ I, A^T A ≈ I  →  True
alpha = -2.5000  →  A A^T ≈ I, A^T A ≈ I  →  True


## Extra Task 7 — Egenvärden för tridiagonal matris

Diagonal 4, sub/super-diagonal 1. För en sådan symmetrisk Toeplitz-matris
av storlek $n$ är egenvärdena
$\lambda_k = 4 + 2\cos\frac{k\pi}{n+1}$, $k = 1, \dots, n$.


In [7]:
n = 20
A = 4 * np.eye(n) + np.diag(np.ones(n-1), 1) + np.diag(np.ones(n-1), -1)
eigs, _ = np.linalg.eig(A)
eigs = np.sort(eigs.real)
analytic = np.sort(4 + 2 * np.cos(np.arange(1, n+1) * np.pi / (n+1)))

print("Numeriska egenvärden:")
print(np.round(eigs, 6))
print("\nMax |numerisk - analytisk| =", np.max(np.abs(eigs - analytic)))


Numeriska egenvärden:
[2.022338 2.088854 2.198062 2.347522 2.533896 2.75302  3.       3.269318
 3.554958 3.85054  4.14946  4.445042 4.730682 5.       5.24698  5.466104
 5.652478 5.801938 5.911146 5.977662]

Max |numerisk - analytisk| = 1.0658141036401503e-14


## Extra Task 8 — Subdiagonal blir $-1$

Matrisen är inte längre symmetrisk → egenvärdena kan bli komplexa. För
denna specifika konstruktion blir produkten av sub- och superdiagonal
negativ ($1 \cdot -1 = -1$); det betyder att de gemensamma egenvärdena är
$4 + 2 i \cos(k\pi/(n+1))$, dvs **rent komplexa förskjutningar** kring 4.


In [8]:
B = 4 * np.eye(n) + np.diag(np.ones(n-1), 1) + np.diag(-np.ones(n-1), -1)
eigs_B, _ = np.linalg.eig(B)
print("Egenvärden av B (sorterat på imaginärdel):")
for ev in sorted(eigs_B, key=lambda z: z.imag):
    print(f"  {ev.real:+.6f}  {ev.imag:+.6f}j")


Egenvärden av B (sorterat på imaginärdel):
  +4.000000  -1.977662j
  +4.000000  -1.911146j
  +4.000000  -1.801938j
  +4.000000  -1.652478j
  +4.000000  -1.466104j
  +4.000000  -1.246980j
  +4.000000  -1.000000j
  +4.000000  -0.730682j
  +4.000000  -0.445042j
  +4.000000  -0.149460j
  +4.000000  +0.149460j
  +4.000000  +0.445042j
  +4.000000  +0.730682j
  +4.000000  +1.000000j
  +4.000000  +1.246980j
  +4.000000  +1.466104j
  +4.000000  +1.652478j
  +4.000000  +1.801938j
  +4.000000  +1.911146j
  +4.000000  +1.977662j
